# Experiment 030b: N=4 group-size ablation, seeds 1 and 2 (Colab)

I extend the seed-0 N=4 versus N=8 comparison to the two remaining seeds.
Each N=4 arm starts from the exact seed-matched merged model used by its 013
n=8 reference. The comparison uses the same 500 optimiser steps but N = 4 samples
half as many completions over the run.

The hard tier remains the primary endpoint. Since seed 0 is unconvincing, the three-seed result can show stability or a
material counterexample, but it cannot establish formal common equivalence.

One Run-all per session. Completed or concluded arms are verified and skipped.


## 0. Compute units and mount Drive


In [1]:
# record the balance before the session starts
import glob, json, os, time
session_id = globals().get('session_id', time.strftime('%Y%m%d_%H%M%S'))
cu_before = 619.1      # fill from the resources panel
unit_rate = 6.77      # units per hour for this card
print('session', session_id, '| cu_before =', cu_before if cu_before is not None else 'FILL ME')

from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/inm363'
RESULTS = os.path.join(ROOT, 'results')
OPEN_UNITS = os.path.join(RESULTS, 'open_cu_sessions')
UNITS_LOG = os.path.join(RESULTS, 'colab_units.jsonl')
os.makedirs(OPEN_UNITS, exist_ok=True)

# close an unfinished session with the balance captured at this runtime start
old_sessions = [p for p in glob.glob(os.path.join(OPEN_UNITS, '*.json'))
                if not p.endswith(f'{session_id}.json')]
open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
if cu_before is None:
    print('reminder: fill cu_before, then rerun this cell to save the session')
else:
    for old_path in old_sessions:
        with open(old_path) as f:
            old = json.load(f)
        old['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        old['colab_cu_after'] = cu_before
        old['colab_cu_used'] = round(old['colab_cu_before'] - cu_before, 2)
        old['closed_on_resume'] = True
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(old) + '\n')
        os.remove(old_path)
        print('closed previous session:', old['session_id'])

    if not os.path.exists(open_path):
        start = {'session_id': session_id, 'ts_start': time.strftime('%Y-%m-%d %H:%M'),
                 'notebook': '030_colab_b7_n4_seeds12',
                 'colab_cu_before': cu_before, 'unit_rate': unit_rate}
        with open(open_path, 'w') as f:
            json.dump(start, f)
        print('opened session record')


session 20260812_225739 | cu_before = 619.1
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
closed previous session: 20260812_225523
opened session record


## 1. GPU check: A100 with 80GB required


In [2]:
import subprocess, torch
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
print(gpu.strip() or 'no gpu visible')
total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'total vram: {total_gb:.1f} GiB')
assert 'A100' in gpu and total_gb >= 79, 'need the a100 80gb runtime used by the 013 reference arms'


GPU 0: NVIDIA A100-SXM4-80GB (UUID: GPU-1666c911-4776-d43e-dc35-ef930ca4c70a)
total vram: 79.3 GiB


## 2. Clone or update the repo


In [3]:
import base64
import getpass
import glob
import os
from pathlib import Path
import subprocess

REPO_SLUG = 'hastov-lab/rl-grpo-or-sft-for-llms'
repo_url = f'https://github.com/{REPO_SLUG}.git'
repo_root = Path('/content') / REPO_SLUG.split('/')[-1]


def repo_git(*args):
    git = ['git']
    env = None
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    # Private forks can use a read token from Colab Secrets.
    if userdata is not None and REPO_SLUG != 'hastov-lab/rl-grpo-or-sft-for-llms':
        try:
            token = userdata.get('GH_TOKEN')
        except Exception:
            token = None
        if isinstance(token, dict):
            token = token.get('value')
        if not isinstance(token, str) or not token.strip():
            token = getpass.getpass('GitHub token: ')
        if not token.strip():
            raise RuntimeError('This private fork requires a GitHub token with read access.')

        # Keep the token out of the command line and the saved remote URL.
        auth = base64.b64encode(f'x-access-token:{token.strip()}'.encode()).decode()
        env = dict(os.environ, REPO_GIT_AUTH=f'Authorization: Basic {auth}')
        git.append('--config-env=http.https://github.com/.extraHeader=REPO_GIT_AUTH')

    subprocess.run([*git, *args], env=env, check=True)


if repo_root.exists():
    if not (repo_root / '.git').exists():
        raise RuntimeError(f'Expected a Git checkout at {repo_root}.')
    origin = subprocess.check_output(
        ['git', '-C', str(repo_root), 'config', '--local', '--get', 'remote.origin.url'], text=True
    ).strip()
    if origin != repo_url:
        raise RuntimeError(f'The checkout at {repo_root} belongs to a different repository.')
    branch = subprocess.check_output(
        ['git', '-C', str(repo_root), 'branch', '--show-current'], text=True
    ).strip()
    if branch != 'main':
        raise RuntimeError(f'Expected the main branch at {repo_root}.')
    repo_git('-C', str(repo_root), 'pull', '--ff-only', 'origin', 'main')
else:
    repo_git('clone', '--branch', 'main', '--single-branch', '--depth', '1', repo_url, str(repo_root))

# Use the scripts from this checkout even if another experiment is open.
code_root = repo_root / '001_Code'
if not (code_root / 'scripts' / 'run.py').is_file():
    raise RuntimeError(f'Experiment scripts are missing from {code_root}.')
os.chdir(code_root)
print('working in:', os.getcwd())


Cloning into 'INM363-IndividualProject-Dissertation'...
remote: Enumerating objects: 3827, done.
remote: Counting objects: 100% (153/153), done.
remote: Compressing objects: 100% (114/114), done.
remote: Total 3827 (delta 57), reused 126 (delta 39), pack-reused 3674 (from 1)
Receiving objects: 100% (3827/3827), 114.98 MiB | 18.39 MiB/s, done.
Resolving deltas: 100% (2456/2456), done.
Updating files: 100% (1775/1775), done.
working in: /content/INM363-IndividualProject-Dissertation/001_Code


## 3. Install dependencies


In [4]:
!pip install -q trl==1.7.1 transformers==5.5.4 peft==0.19.1 accelerate==1.14.0 datasets==4.8.4 wandb pyyaml


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.4/842.4 kB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 162.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 53.8 MB/s eta 0:00:00


## 4. W&B key


In [5]:
import getpass, os
key = None
try:
    from google.colab import userdata
    key = userdata.get('WANDB_API_KEY')
except Exception:
    key = None
if not key and os.path.exists('wandb_api_key.txt'):
    key = open('wandb_api_key.txt').read().strip()
if not key:
    key = getpass.getpass('wandb api key: ').strip()
assert key, 'a wandb key is required because the worker logs online'
os.environ['WANDB_API_KEY'] = key
with open('wandb_api_key.txt', 'w') as f:
    f.write(key)
key = None
print('wandb key set')


wandb key set


## 5. Arm selection


In [6]:
SEEDS_TO_RUN = (1, 2)
assert set(SEEDS_TO_RUN).issubset({1, 2})
print('seeds this session:', SEEDS_TO_RUN)


seeds this session: (1, 2)


## 6. Preflight before any test-set contact


In [7]:
import json, os, shutil, sys
sys.path.insert(0, os.getcwd())
from config import load_config, run_tag
from scripts.s12_eval import atomic_write_json, config_hash_status, read_jsonl, sha256
from scripts.s28_compose import (check_merged_complete, merged_hashes,
                                 resolve_source_adapter, verify_historical_merge)

D = '/content/drive/MyDrive/inm363'
ARMS_030B = [
    ('configs/030_colab_b7_n4_seed1.yaml', 1,
     '013_COLAB_FULL_B7_sft_grpo_seed1',
     'results/022_smooth_B7_seed1_step1000/arm_manifest.json'),
    ('configs/030_colab_b7_n4_seed2.yaml', 2,
     '013_COLAB_FULL_B7_sft_grpo_seed2',
     'results/022_smooth_B7_seed2_step1000/arm_manifest.json'),
]
INVENTORY = (
    'sft_eval.json', 'sft_preds.jsonl', 'sft_logprob.json',
    'sft_logprob_preds.jsonl', 'grpo_eval.json', 'grpo_preds.jsonl',
    'grpo_logprob.json', 'grpo_logprob_preds.jsonl',
    'grpo_trainlog.json', 'manifest.json')

CFG, TAG, PARENT_TAG, PARENT_RESULTS = {}, {}, {}, {}
ADIR, ADAPTER_HASHES, ARCHIVE_MERGED, HISTORICAL = {}, {}, {}, {}
problems = []
for cfg_path, seed, parent_tag, historical_manifest in ARMS_030B:
    cfg = load_config(cfg_path)
    assert cfg.experiment == '030' and cfg.train.seed == seed
    assert cfg.condition.name == 'EXTERNAL_SFT_GRPO'
    assert cfg.train.num_generations == 4
    CFG[seed], TAG[seed], PARENT_TAG[seed] = cfg, run_tag(cfg), parent_tag
    parent_results = os.path.join(D, 'results', parent_tag)
    archive_merged = os.path.join(D, 'checkpoints', parent_tag, 'sft_merged')
    PARENT_RESULTS[seed], ARCHIVE_MERGED[seed] = parent_results, archive_merged

    try:
        adir, adapter_hashes = resolve_source_adapter(cfg.train.grpo_from_adapter, D)
        ADIR[seed], ADAPTER_HASHES[seed] = adir, adapter_hashes
        parent_cfg = load_config(f'configs/013_colab_full_b7_seed{seed}.yaml')
        parent_man = json.load(open(os.path.join(parent_results, 'manifest.json')))
        stored = open(os.path.join(adir, 'config_hash.txt')).read().strip()
        hash_status = config_hash_status(parent_cfg, parent_man, stored)
        if hash_status not in ('current', 'legacy_missing_grpo_from_adapter'):
            problems.append(f'seed {seed}: parent adapter hash status is {hash_status}')

        artifact_hashes = {}
        for name in INVENTORY:
            path = os.path.join(parent_results, name)
            if not os.path.exists(path):
                problems.append(f'seed {seed}: 013 parent missing {name}')
                continue
            artifact_hashes[name] = sha256(path)
            tracked_path = os.path.join('results', parent_tag, name)
            if not os.path.exists(tracked_path):
                problems.append(f'seed {seed}: tracked 013 reference missing {name}')
            elif sha256(tracked_path) != artifact_hashes[name]:
                problems.append(f'seed {seed}: Drive {name} differs from the tracked 013 reference')
        for metrics_name, preds_name in (
                ('sft_eval.json', 'sft_preds.jsonl'),
                ('sft_logprob.json', 'sft_logprob_preds.jsonl'),
                ('grpo_eval.json', 'grpo_preds.jsonl'),
                ('grpo_logprob.json', 'grpo_logprob_preds.jsonl')):
            metrics_path = os.path.join(parent_results, metrics_name)
            preds_path = os.path.join(parent_results, preds_name)
            if os.path.exists(metrics_path) and os.path.exists(preds_path):
                assert json.load(open(metrics_path)).get('n_items') == 1884,                     f'seed {seed}: {metrics_name} is incomplete'
                assert len(read_jsonl(preds_path)) == 1884,                     f'seed {seed}: {preds_name} is incomplete'

        check_merged_complete(archive_merged)
        archive_hashes = merged_hashes(archive_merged)
        historical = verify_historical_merge(
            archive_merged, historical_manifest, parent_tag)
        HISTORICAL[seed] = historical
        record = {'config': cfg_path, 'tag': TAG[seed], 'parent_tag': parent_tag,
                  'adapter_sha256': adapter_hashes,
                  'parent_artifact_sha256': artifact_hashes,
                  'archive_source': archive_merged,
                  'archive_sha256': archive_hashes,
                  'historical_merge': historical,
                  'parent_hash_status': hash_status}
        pf_path = os.path.join(D, 'results', f'030b_preflight_seed{seed}.json')
        if os.path.exists(pf_path):
            assert json.load(open(pf_path)) == record,                 f'seed {seed}: preflight sources changed, halting'
        else:
            atomic_write_json(pf_path, record)
        print(f'seed {seed}: preflight recorded and verified')
    except Exception as e:
        problems.append(f'seed {seed}: {e}')

free_gb = shutil.disk_usage(D).free / 1024**3
print(f'drive free: {free_gb:.0f} GB')
if free_gb < 60:
    problems.append(f'only {free_gb:.0f}GB free on drive, two copied 7b bases and checkpoints need 60')
assert not problems, 'preflight halt:\n' + '\n'.join(problems)
print('preflight green for seeds 1 and 2')


seed 1: preflight recorded and verified
seed 2: preflight recorded and verified
drive free: 151 GB
preflight green for seeds 1 and 2


## 7. Wiring checks


In [8]:
import subprocess, sys
r = subprocess.run([sys.executable, '-m', 'pytest',
                    'tests/test_030_n4_ablation.py', 'tests/test_s28_composition.py',
                    'tests/test_checkpoint.py', 'tests/test_grpo_worker.py', '-q'])
assert r.returncode == 0, 'package tests failed, nothing runs'

from scripts.grpo_worker import (active_worker_processes, blocked_resume_reason,
                                 stream_subprocess)
from scripts.s12_eval import IntegrityError, identity_check
from scripts.s28_compose import (arm_state, commit_endpoint_eval,
                                 commit_training_complete, conclude_a2,
                                 concluded_a2_state, endpoint_state, gate_state,
                                 sentinel_abort_reason, stage_archived_merge,
                                 status_is_a2, training_state,
                                 verify_archived_merge, write_arm_complete)
from obs import _versions, append_ledger, gpu_snapshot, _git_info
from train.model_utils import resolve_dtype
print('package imports fine')


package imports fine


In [9]:
!pip uninstall -q -y torchao


## 8. Build the data splits


In [10]:
!python scripts/build_data.py


README.md: 100% 7.93k/7.93k [00:00<00:00, 24.2MB/s]

main/train-00000-of-00001.parquet: downloading bytes:   0% 8.52k/2.31M [00:01<06:01, 6.36kB/s]
main/train-00000-of-00001.parquet: downloading bytes: 100% 2.30M/2.30M [00:01<00:00, 1.63MB/s,  224kB/s  ]
main/train-00000-of-00001.parquet: reconstructing file: 100% 2.31M/2.31M [00:01<00:00, 1.64MB/s,  224kB/s  ]

main/test-00000-of-00001.parquet: downloading bytes:   0% 0.00/419k [00:00<?, ?B/s]
main/test-00000-of-00001.parquet: downloading bytes: 100% 419k/419k [00:01<00:00, 334kB/s, 40.8kB/s  ]
main/test-00000-of-00001.parquet: reconstructing file: 100% 419k/419k [00:01<00:00, 334kB/s, 40.8kB/s  ]
Generating train split: 100% 7473/7473 [00:00<00:00, 381499.92 examples/s]
Generating test split: 100% 1319/1319 [00:00<00:00, 319434.55 examples/s]
README.md: 100% 53.2k/53.2k [00:00<00:00, 93.9MB/s]
dataset_infos.json: 100% 138k/138k [00:00<00:00, 150MB/s]

high_school_psychology/test-00000-of-000(…): downloading bytes:   0% 0.00/92.8k [00

## 9. Reuse the archived base evaluation


In [11]:
from scripts.s28_compose import prepopulate_base_evals
src = os.path.join(D, 'results', 'base_evals', '013_COLAB_FULL_B7')
dst = os.path.join(D, 'results', 'base_evals', '030_COLAB_FULL_B7')
base_record = prepopulate_base_evals(src, dst)
print('base reuse covers', sorted(base_record['sha256']))


  base evals already prepopulated and verified
base reuse covers ['base_eval.json', 'base_logprob.json', 'base_logprob_preds.jsonl', 'base_preds.jsonl']


## 10. Copy the exact seed-matched merged models


In [12]:
OUT_DIR, MERGED_DIR, RES_DIR = {}, {}, {}
for seed in SEEDS_TO_RUN:
    OUT_DIR[seed] = os.path.join(D, 'checkpoints', TAG[seed], 'grpo')
    MERGED_DIR[seed] = os.path.join(D, 'checkpoints', TAG[seed], 'sft_merged')
    RES_DIR[seed] = os.path.join(D, 'results', TAG[seed])
    verify_historical_merge(
        ARCHIVE_MERGED[seed], HISTORICAL[seed]['manifest'], PARENT_TAG[seed])
    record = stage_archived_merge(
        ARCHIVE_MERGED[seed], MERGED_DIR[seed], ADIR[seed], OUT_DIR[seed])
    verify_archived_merge(
        ARCHIVE_MERGED[seed], MERGED_DIR[seed], ADIR[seed])
    print(f'seed {seed}: copied files', len(record['destination_sha256']))


  archived 013 merged base copied and committed
seed 1: copied files 6
  archived 013 merged base copied and committed
seed 2: copied files 6


## 11. Parent identity probes


In [13]:
import shutil as _shutil, subprocess, sys, time

GATE_SCRIPT = '/content/s30b_gate.py'
with open(GATE_SCRIPT, 'w') as f:
    f.write('''import argparse, os, sys
sys.path.insert(0, os.getcwd())
from config import load_config
from scripts.env_switch import apply_smoke, resolve_device
from eval.harness import run_logprob_eval

ap = argparse.ArgumentParser()
ap.add_argument("--config", required=True)
ap.add_argument("--adapter", required=True)
ap.add_argument("--out", required=True)
args = ap.parse_args()
cfg = apply_smoke(load_config(args.config))
run_logprob_eval(cfg, resolve_device(cfg.env), adapter_path=args.adapter,
                 condition="gate", out_dir=args.out)
''')

for cfg_path, seed, parent_tag, _ in ARMS_030B:
    res_dir = os.path.join(D, 'results', TAG[seed])
    os.makedirs(res_dir, exist_ok=True)
    gate_path = os.path.join(res_dir, 'parent_gate.json')
    archived_preds = os.path.join(PARENT_RESULTS[seed], 'sft_logprob_preds.jsonl')
    if gate_state(gate_path, ADAPTER_HASHES[seed], archived_preds) == 'committed':
        print(f'seed {seed}: parent gate already committed, skipping')
        continue
    gate_dir = f'/content/s30b_gate_seed{seed}'
    gate_preds = os.path.join(gate_dir, 'gate_logprob_preds.jsonl')
    rows_ok = (os.path.exists(gate_preds)
               and sum(1 for line in open(gate_preds) if line.strip()) == 1884)
    if not rows_ok:
        _shutil.rmtree(gate_dir, ignore_errors=True)
        t0 = time.time()
        r = subprocess.run([sys.executable, GATE_SCRIPT, '--config', cfg_path,
                            '--adapter', ADIR[seed], '--out', gate_dir])
        assert r.returncode == 0, f'seed {seed}: parent identity probe failed'
        elapsed = round(time.time() - t0, 1)
    else:
        elapsed = None
    new_rows = read_jsonl(gate_preds)
    assert len(new_rows) == 1884, f'seed {seed}: identity probe scratch is incomplete'
    stats = identity_check(new_rows, read_jsonl(archived_preds))
    atomic_write_json(gate_path, {
        'ts': time.strftime('%Y-%m-%d %H:%M'), 'completed': True,
        'parent': parent_tag, 'adapter_sha256': ADAPTER_HASHES[seed],
        'archived_preds_sha256': sha256(archived_preds),
        'new_preds_sha256': sha256(gate_preds),
        'stats': stats, 'elapsed_sec': elapsed})
    _shutil.rmtree(gate_dir, ignore_errors=True)
    print(f'seed {seed}: parent gate', stats)

free_gib = torch.cuda.mem_get_info()[0] / 1024**3
print(f'gpu free: {free_gib:.1f} GiB')
assert free_gib >= 70, 'need 70 GiB free before the first 7b training run'


seed 1: parent gate {'agreement': 1.0, 'acc_new': 0.801486, 'acc_archived': 0.801486}
seed 2: parent gate {'agreement': 1.0, 'acc_new': 0.798301, 'acc_archived': 0.798301}
gpu free: 78.8 GiB


## 12. Train and evaluate N=4


In [14]:
assert cu_before is not None and unit_rate is not None, 'fill cu_before and unit_rate in the units cell'
import glob as _glob, subprocess, sys, time

EVAL_SCRIPT = '/content/s30b_eval.py'
with open(EVAL_SCRIPT, 'w') as f:
    f.write('''import argparse, os, sys
sys.path.insert(0, os.getcwd())
from checkpoint import save_metrics
from config import load_config
from scripts.env_switch import apply_smoke, resolve_device
from eval.harness import run_eval, run_logprob_eval

ap = argparse.ArgumentParser()
ap.add_argument("--config", required=True)
ap.add_argument("--adapter", required=True)
ap.add_argument("--out", required=True)
args = ap.parse_args()
cfg = apply_smoke(load_config(args.config))
device = resolve_device(cfg.env)
ev = run_eval(cfg, device, adapter_path=args.adapter, condition="grpo", out_dir=args.out)
save_metrics(os.path.join(args.out, "grpo_eval.json"), ev)
lp = run_logprob_eval(cfg, device, adapter_path=args.adapter, condition="grpo", out_dir=args.out)
save_metrics(os.path.join(args.out, "grpo_logprob.json"), lp)
''')

os.environ['WANDB_PROJECT'] = 'inm363-grpo'
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
LEDGER = os.path.join(D, 'results', 'compute_ledger.jsonl')
allowed_infra_files = {
    'notebooks/030_colab_b7_n4_seeds12.ipynb',
    'scripts/grpo_worker.py',
    'tests/test_030_n4_ablation.py',
    'tests/test_grpo_worker.py',
}

for cfg_path, seed, parent_tag, historical_manifest in ARMS_030B:
    if seed not in SEEDS_TO_RUN:
        continue
    print('===== ' + TAG[seed] + ' =====')
    cfg = CFG[seed]
    out_dir, merged_dir, res_dir = OUT_DIR[seed], MERGED_DIR[seed], RES_DIR[seed]
    gate_path = os.path.join(res_dir, 'parent_gate.json')

    parent_manifest = json.load(open(os.path.join(PARENT_RESULTS[seed], 'manifest.json')))
    for lib in ('torch', 'trl', 'transformers', 'peft', 'accelerate',
                'cais/mmlu@', 'openai/gsm8k@'):
        want = parent_manifest.get('versions', {}).get(lib)
        if want is not None:
            got = _versions().get(lib)
            assert got == want, f'seed {seed}, 013 {lib}: {got} vs {want}'
    assert resolve_dtype(cfg) == parent_manifest['resolved_dtype'],         f'seed {seed}: dtype differs from 013'
    assert cfg.exec.allow_oom_fallback is False, 'the ladder must stay off'
    assert (cfg.exec.grpo_batch_size or cfg.train.num_generations) == 4
    assert cfg.exec.grpo_grad_accum == 1 and cfg.exec.gradient_checkpointing is False
    assert cfg.log.wandb_project == 'inm363-grpo'
    verify_historical_merge(ARCHIVE_MERGED[seed], historical_manifest, parent_tag)
    verify_archived_merge(ARCHIVE_MERGED[seed], merged_dir, ADIR[seed])

    def newest_step():
        checkpoints = _glob.glob(os.path.join(out_dir, 'checkpoint-*'))
        return max((int(path.rsplit('-', 1)[1]) for path in checkpoints), default=0)

    def save_available_history():
        path = os.path.join(res_dir, 'grpo_trainlog.json')
        if os.path.exists(path):
            return json.load(open(path))
        checkpoints = sorted(_glob.glob(os.path.join(out_dir, 'checkpoint-*')),
                             key=lambda path: int(path.rsplit('-', 1)[1]), reverse=True)
        for checkpoint in checkpoints:
            state_path = os.path.join(checkpoint, 'trainer_state.json')
            if os.path.exists(state_path):
                history = json.load(open(state_path)).get('log_history', [])
                if history:
                    atomic_write_json(path, history)
                    return history
        return []

    final_artifacts = [os.path.join(out_dir, 'adapter_model.safetensors'),
                       os.path.join(out_dir, 'adapter_base.json'),
                       os.path.join(res_dir, 'grpo_trainlog.json')]
    if arm_state(res_dir, out_dir) == 'committed':
        assert training_state(res_dir, out_dir, merged_dir, cfg) == 'committed'
        verify_archived_merge(ARCHIVE_MERGED[seed], merged_dir, ADIR[seed])
        print('arm complete, skipping')
        continue

    man_path = os.path.join(res_dir, 'arm_manifest.json')
    man = json.load(open(man_path)) if os.path.exists(man_path) else {'attempts': []}
    start_step = newest_step()
    for attempt in man.get('attempts', []):
        if 'ts_end' not in attempt:
            attempt.update({'interrupted': True, 'end_step': start_step,
                            'elapsed_sec': None, 'ts_end': 'runtime died before finally'})
    atomic_write_json(man_path, man)

    if concluded_a2_state(res_dir, man.get('attempts', [])) == 'committed':
        print('arm concluded by sentinel a2 previously, skipping')
        continue
    a2_prior = sentinel_abort_reason(man.get('attempts', []))
    if a2_prior:
        history = save_available_history()
        evidence = {'num_generations': 4, 'history_records': len(history),
                    'last_logged_step': max((row.get('step', 0) for row in history), default=0)}
        conclude_a2(res_dir, man['attempts'][-1], evidence=evidence)
        print('repaired the missing a2 conclusion record, next arm')
        continue

    reason = blocked_resume_reason(man.get('attempts', []))
    assert reason is None, f'seed {seed}: {reason}'
    assert not active_worker_processes(cfg_path), 'a training worker is already running'
    trained = all(os.path.exists(path) for path in final_artifacts)
    if not trained:
        env = {'git': _git_info(), 'versions': _versions(),
               'resolved_dtype': resolve_dtype(cfg), 'ladder': 'disabled'}
        repo_top = subprocess.run(['git', 'rev-parse', '--show-toplevel'],
                                  capture_output=True, text=True, check=True).stdout.strip()
        repo_prefix = os.path.relpath(os.getcwd(), repo_top)
        for attempt in man.get('attempts', []):
            previous = attempt.get('env')
            if not previous:
                continue
            old_commit = previous['git'].get('commit')
            new_commit = env['git'].get('commit')
            if old_commit != new_commit:
                changed = subprocess.run(['git', 'diff', '--name-only', old_commit, new_commit],
                                         capture_output=True, text=True,
                                         check=True).stdout.splitlines()
                prefix = repo_prefix + os.sep
                changed = [name[len(prefix):] if repo_prefix != '.' and name.startswith(prefix)
                           else name for name in changed]
                unexpected = sorted(set(changed) - allowed_infra_files)
                assert not unexpected, f'seed {seed}: scientific code drifted: {unexpected}'
            for lib in ('torch', 'trl', 'transformers', 'peft', 'accelerate'):
                assert previous['versions'].get(lib) == env['versions'].get(lib),                     f'seed {seed}: {lib} drifted'
            assert previous['resolved_dtype'] == env['resolved_dtype'],                 f'seed {seed}: dtype drifted'

        free_gib = torch.cuda.mem_get_info()[0] / 1024**3
        assert free_gib >= 70, f'only {free_gib:.1f} GiB free before seed {seed}'
        verify_historical_merge(ARCHIVE_MERGED[seed], historical_manifest, parent_tag)
        verify_archived_merge(ARCHIVE_MERGED[seed], merged_dir, ADIR[seed])
        attempt = {'start_step': start_step, 'ts_start': time.strftime('%Y-%m-%d %H:%M'),
                   'gpu': gpu_snapshot(), 'env': env,
                   'adapter_sha256': ADAPTER_HASHES[seed], 'completed': False}
        man['attempts'].append(attempt)
        atomic_write_json(man_path, man)

        status_path = f'/content/030b_worker_seed{seed}.json'
        if os.path.exists(status_path):
            os.remove(status_path)
        worker_cmd = [sys.executable, '-u', 'scripts/grpo_worker.py',
                      '--config', cfg_path, '--adapter', ADIR[seed],
                      '--out-dir', out_dir, '--log-dir', res_dir,
                      '--run-name', f'{TAG[seed]}_grpo', '--status', status_path,
                      '--report-to', 'wandb']
        t0 = time.time()
        proc = None
        parent_failure = None
        a2_concluded = False
        try:
            print(f'===== seed {seed}, N=4 from step {start_step} =====')
            proc = stream_subprocess(worker_cmd)
            if proc.returncode != 0:
                status_now = json.load(open(status_path)) if os.path.exists(status_path) else {}
                if status_is_a2(status_now):
                    a2_concluded = True
                else:
                    raise RuntimeError(
                        f'seed {seed}: training worker failed with exit code {proc.returncode}')
        except BaseException as exc:
            parent_failure = {'error_type': type(exc).__name__,
                              'error_message': str(exc)[:2000]}
            raise
        finally:
            wall = time.time() - t0
            end_step = newest_step()
            worker_returncode = proc.returncode if proc is not None else None
            worker_status = json.load(open(status_path)) if os.path.exists(status_path) else {}
            worker_peak = worker_status.get('peak_vram_gib')
            error_type = worker_status.get('error_type') or (parent_failure or {}).get('error_type')
            error_message = worker_status.get('error_message') or (parent_failure or {}).get('error_message')
            manual_interrupt = error_type == 'KeyboardInterrupt'
            explicit_failure = ((worker_returncode not in (None, 0) or parent_failure is not None)
                                and not manual_interrupt)
            done = (worker_status.get('completed') is True
                    and all(os.path.exists(path) for path in final_artifacts))
            attempt.update({'end_step': end_step, 'ts_end': time.strftime('%Y-%m-%d %H:%M'),
                            'elapsed_sec': round(wall, 1), 'completed': done,
                            'peak_vram_gib': worker_peak, 'worker_returncode': worker_returncode,
                            'error_type': error_type, 'error_message': error_message,
                            'cuda_oom': bool(worker_status.get('cuda_oom', False)),
                            'interrupted': manual_interrupt, 'resume_blocked': explicit_failure})
            atomic_write_json(man_path, man)
            gpu_row = gpu_snapshot()
            if worker_peak is not None:
                gpu_row['peak_vram_gb'] = worker_peak
            append_ledger(LEDGER, {'run_tag': TAG[seed],
                                   'condition': 'EXTERNAL_SFT_GRPO_train',
                                   'timings_sec': {'grpo_train': round(wall, 1)},
                                   'train_steps': end_step - start_step,
                                   'completed': attempt['completed'], 'gpu': gpu_row})
        if a2_concluded:
            history = save_available_history()
            evidence = {'num_generations': 4, 'history_records': len(history),
                        'last_logged_step': max((row.get('step', 0) for row in history), default=0)}
            conclude_a2(res_dir, man['attempts'][-1], evidence=evidence)
            print('concluded by exact sentinel a2, next arm')
            continue
        assert worker_status.get('completed') is True, 'worker status did not complete'
        assert attempt['completed'] is True, 'final training artifacts are incomplete'
        print(f'trained in {wall/3600:.2f}h, peak {worker_peak} GiB')
    else:
        print('adapter already trained')

    if training_state(res_dir, out_dir, merged_dir, cfg) == 'absent':
        commit_training_complete(res_dir, out_dir, merged_dir, cfg)
    assert training_state(res_dir, out_dir, merged_dir, cfg) == 'committed'
    verify_historical_merge(ARCHIVE_MERGED[seed], historical_manifest, parent_tag)
    verify_archived_merge(ARCHIVE_MERGED[seed], merged_dir, ADIR[seed])
    _, hashes_now = resolve_source_adapter(cfg.train.grpo_from_adapter, D)
    assert hashes_now == ADAPTER_HASHES[seed], 'source adapter drifted since preflight'

    ep_state = endpoint_state(res_dir, out_dir)
    if ep_state == 'tampered':
        raise IntegrityError(f'seed {seed}: endpoint files contradict their manifest')
    if ep_state != 'committed':
        scratch = f'/content/s30b_eval_seed{seed}'
        _shutil.rmtree(scratch, ignore_errors=True)
        t0 = time.time()
        eval_done = False
        try:
            r = subprocess.run([sys.executable, EVAL_SCRIPT, '--config', cfg_path,
                                '--adapter', out_dir, '--out', scratch])
            assert r.returncode == 0, f'seed {seed}: endpoint evaluation failed'
            eval_wall = round(time.time() - t0, 1)
            gate_record = json.load(open(gate_path))
            extra = {'adapter': out_dir, 'source_adapter': ADIR[seed],
                     'source_adapter_sha256': ADAPTER_HASHES[seed],
                     'archived_merge': verify_archived_merge(
                         ARCHIVE_MERGED[seed], merged_dir, ADIR[seed]),
                     'historical_merge': verify_historical_merge(
                         ARCHIVE_MERGED[seed], historical_manifest, parent_tag),
                     'parent_gate_stats': gate_record['stats'],
                     'training_complete': json.load(open(
                         os.path.join(res_dir, 'training_complete.json'))),
                     'config': cfg_path, 'seed': seed, 'git': _git_info(),
                     'versions': _versions(), 'resolved_dtype': resolve_dtype(cfg),
                     'gpu': gpu_snapshot(), 'timings_sec': {'grpo_eval': eval_wall}}
            commit_endpoint_eval(res_dir, scratch, adapter_dir=out_dir,
                                 extra_manifest=extra, package='030_n4')
            _shutil.rmtree(scratch, ignore_errors=True)
            eval_done = True
        finally:
            append_ledger(LEDGER, {'run_tag': TAG[seed],
                                   'condition': 'EXTERNAL_SFT_GRPO_eval',
                                   'timings_sec': {'grpo_eval': round(time.time() - t0, 1)},
                                   'train_steps': 0, 'completed': eval_done,
                                   'gpu': gpu_snapshot()})

    r = subprocess.run([sys.executable, 'scripts/run.py', '--config', cfg_path,
                        '--stage', 'all', '--no-ledger'])
    assert r.returncode == 0, f'seed {seed}: summary stage failed'
    summary = json.load(open(os.path.join(res_dir, 'summary.json')))
    assert summary['reference'] == f'gsm8k_sft_parent:{parent_tag}'
    assert training_state(res_dir, out_dir, merged_dir, cfg) == 'committed'
    write_arm_complete(res_dir)
    print('gen per-tier:', summary['grpo_acc'])

print('session done for seeds', SEEDS_TO_RUN)


Streaming output truncated to the last 5000 lines.
│ │ its despotic  │               │               │              │           │ │
│ │ character,    │               │               │              │           │ │
│ │ which is      │               │               │              │           │ │
│ │ rooted in a   │               │               │              │           │ │
│ │ sense of its  │               │               │              │           │ │
│ │ special       │               │               │              │           │ │
│ │ mission       │               │               │              │           │ │
│ │ <think>       │               │               │              │           │ │
│ └───────────────┴───────────────┴───────────────┴──────────────┴───────────┘ │
╰──────────────────────────────────────────────────────────────────────────────╯
{'time': '02:38:40', 'loss': '0.0004038', 'grad_norm': '0.01671', 'learning_rate': '5e-07', 'num_tokens': '3.891e+05', 'completions/mean_le

## 13. Three-seed comparison


In [15]:
assert unit_rate is not None, 'fill unit_rate in the units cell'
import statistics
from analysis.stats import _read_preds, align_pred_rows, holm, mcnemar_exact, paired_delta_ci
from data.subjects import TIERS, tier_of

def metric_summary(history, key):
    values = [(int(row['step']), float(row[key])) for row in history
              if row.get('step') is not None and row.get(key) is not None]
    if not values:
        return None
    first = [value for step, value in values if step <= 50]
    last = [value for step, value in values if step > 450]
    return {'mean': statistics.fmean(value for _, value in values),
            'first_50': statistics.fmean(first) if first else None,
            'last_50': statistics.fmean(last) if last else None}

def dynamics(path):
    history = json.load(open(path))
    keys = ('frac_reward_zero_std', 'reward_std',
            'rewards/accuracy_reward/mean', 'rewards/format_reward/mean',
            'kl', 'completions/mean_length', 'step_time')
    out = {key: metric_summary(history, key) for key in keys}
    out['total_generated_tokens'] = max(
        (float(row['num_tokens']) for row in history if row.get('num_tokens') is not None),
        default=None)
    out['logged_steps'] = max((int(row.get('step', 0)) for row in history), default=0)
    return out

def classify(delta, lo, hi, adjusted_p):
    if abs(delta) >= 0.02 and adjusted_p < 0.05:
        return 'material improvement' if delta > 0 else 'material decline'
    if lo > -0.02 and hi < 0.02:
        return 'equivalent'
    return 'inconclusive'

pairs = []
for seed in (0, 1, 2):
    cfg_path = f'configs/030_colab_b7_n4_seed{seed}.yaml'
    cfg = load_config(cfg_path)
    tag = run_tag(cfg)
    parent_tag = f'013_COLAB_FULL_B7_sft_grpo_seed{seed}'
    res_dir = os.path.join(D, 'results', tag)
    out_dir = os.path.join(D, 'checkpoints', tag, 'grpo')
    merged_dir = os.path.join(D, 'checkpoints', tag, 'sft_merged')
    parent_dir = os.path.join(D, 'results', parent_tag)
    man_path = os.path.join(res_dir, 'arm_manifest.json')
    attempts = json.load(open(man_path)).get('attempts', []) if os.path.exists(man_path) else []

    if seed == 0:
        for ref_tag, names in (
                (parent_tag, ('grpo_preds.jsonl', 'grpo_logprob_preds.jsonl',
                              'grpo_trainlog.json', 'manifest.json')),
                (tag, ('grpo_preds.jsonl', 'grpo_logprob_preds.jsonl',
                       'grpo_trainlog.json', 'manifest.json', 'grpo_eval_manifest.json'))):
            for name in names:
                tracked = os.path.join('results', ref_tag, name)
                drive_copy = os.path.join(D, 'results', ref_tag, name)
                assert os.path.exists(tracked) and os.path.exists(drive_copy),                     f'seed 0 reference missing {ref_tag}/{name}'
                assert sha256(tracked) == sha256(drive_copy),                     f'seed 0 reference drifted: {ref_tag}/{name}'

    if concluded_a2_state(res_dir, attempts) == 'committed':
        pairs.append({'seed': seed, 'status': 'viability_gate_failure',
                      'a2_record': json.load(open(os.path.join(res_dir, 'concluded_a2.json')))})
        continue
    if arm_state(res_dir, out_dir) != 'committed':
        pairs.append({'seed': seed, 'status': 'not_completed'})
        continue
    assert training_state(res_dir, out_dir, merged_dir, cfg) == 'committed'

    n8_rows = _read_preds(os.path.join(parent_dir, 'grpo_preds.jsonl'))
    n4_rows = _read_preds(os.path.join(res_dir, 'grpo_preds.jsonl'))
    a8, a4 = align_pred_rows(n8_rows, n4_rows)
    tiers = {}
    for tier in TIERS:
        rows = [(old['correct'], new['correct']) for old, new in zip(a8, a4)
                if tier_of(old['subject']) == tier]
        old_correct = [old for old, _ in rows]
        new_correct = [new for _, new in rows]
        delta, lo, hi = paired_delta_ci(old_correct, new_correct, seed=seed)
        tiers[tier] = {'n8_acc': sum(old_correct) / len(old_correct),
                       'n4_acc': sum(new_correct) / len(new_correct),
                       'delta': delta, 'ci': [lo, hi]}

    hard_old = [old['correct'] for old, new in zip(a8, a4)
                if tier_of(old['subject']) == 'hard']
    hard_new = [new['correct'] for old, new in zip(a8, a4)
                if tier_of(old['subject']) == 'hard']
    n4_gain = sum(1 for old, new in zip(hard_old, hard_new) if not old and new)
    n8_gain = sum(1 for old, new in zip(hard_old, hard_new) if old and not new)
    raw_p = mcnemar_exact(n4_gain, n8_gain)

    overall_old = [row['correct'] for row in a8]
    overall_new = [row['correct'] for row in a4]
    delta, lo, hi = paired_delta_ci(overall_old, overall_new, seed=seed)
    tiers['overall'] = {'n8_acc': sum(overall_old) / len(overall_old),
                        'n4_acc': sum(overall_new) / len(overall_new),
                        'delta': delta, 'ci': [lo, hi]}

    p8 = _read_preds(os.path.join(parent_dir, 'grpo_logprob_preds.jsonl'))
    p4 = _read_preds(os.path.join(res_dir, 'grpo_logprob_preds.jsonl'))
    pa8, pa4 = align_pred_rows(p8, p4)
    probe_delta, probe_lo, probe_hi = paired_delta_ci(
        [row['correct'] for row in pa8], [row['correct'] for row in pa4], seed=seed)
    probe_category = ('equivalent' if probe_lo > -0.02 and probe_hi < 0.02
                      else 'equivalence not established')

    n4_attempt = attempts[-1] if attempts else {}
    n8_manifest = json.load(open(os.path.join(parent_dir, 'manifest.json')))
    endpoint_manifest = json.load(open(os.path.join(res_dir, 'grpo_eval_manifest.json')))
    train_seconds = n4_attempt.get('elapsed_sec')
    eval_seconds = endpoint_manifest.get('timings_sec', {}).get('grpo_eval')
    core_seconds = sum(value for value in (train_seconds, eval_seconds) if value is not None)
    pairs.append({
        'seed': seed, 'status': 'completed', 'hard_raw_p': raw_p,
        'hard_discordant': {'n4_gain': n4_gain, 'n8_gain': n8_gain},
        'tiers': tiers,
        'probe': {'delta': probe_delta, 'ci': [probe_lo, probe_hi],
                  'category': probe_category},
        'format_rate': {
            'n8': sum(row.get('format_ok', False) for row in a8) / len(a8),
            'n4': sum(row.get('format_ok', False) for row in a4) / len(a4)},
        'mean_response_tokens': {
            'n8': statistics.fmean(row.get('n_tokens', 0) for row in a8),
            'n4': statistics.fmean(row.get('n_tokens', 0) for row in a4)},
        'training_seconds': {
            'n8': n8_manifest.get('timings_sec', {}).get('grpo_train'),
            'n4': train_seconds},
        'evaluation_seconds_n4': eval_seconds,
        'estimated_core_units_n4': core_seconds / 3600 * unit_rate,
        'peak_vram_gib': {
            'n8': n8_manifest.get('gpu', {}).get('peak_vram_gb'),
            'n4': n4_attempt.get('peak_vram_gib')},
        'dynamics': {
            'n8': dynamics(os.path.join(parent_dir, 'grpo_trainlog.json')),
            'n4': dynamics(os.path.join(res_dir, 'grpo_trainlog.json'))}})

completed = [row for row in pairs if row['status'] == 'completed']
binding = len(completed) == 3
if binding:
    adjusted = holm([row['hard_raw_p'] for row in completed])
    for row, adjusted_p in zip(completed, adjusted):
        hard = row['tiers']['hard']
        row['hard_holm_p'] = adjusted_p
        row['hard_category'] = classify(
            hard['delta'], hard['ci'][0], hard['ci'][1], adjusted_p)
else:
    for row in completed:
        row['hard_holm_p'] = None
        row['hard_category'] = None

categories = [row.get('hard_category') for row in completed]
common = None
if binding and len(set(categories)) == 1 and categories[0] != 'inconclusive':
    common = categories[0]
hard_deltas = [row['tiers']['hard']['delta'] for row in completed]
overall_deltas = [row['tiers']['overall']['delta'] for row in completed]
result = {
    'experiment': '030', 'package': '030b',
    'status': 'completed' if binding else 'descriptive_incomplete',
    'primary_endpoint': 'hard', 'holm_binding': binding,
    'common_reading': common,
    'formal_equivalence_possible': False,
    'scope': 'outcome-informed exploratory robustness across seeds 0, 1 and 2',
    'pairs': pairs,
    'descriptive_summary': {
        'hard_delta_mean': statistics.fmean(hard_deltas) if hard_deltas else None,
        'hard_delta_range': [min(hard_deltas), max(hard_deltas)] if hard_deltas else None,
        'overall_delta_mean': statistics.fmean(overall_deltas) if overall_deltas else None,
        'overall_delta_range': [min(overall_deltas), max(overall_deltas)] if overall_deltas else None,
        'all_hard_point_estimates_within_2pt': (
            all(abs(value) < 0.02 for value in hard_deltas) if binding else None)},
}
family_path = os.path.join(D, 'results', '030_family_summary.json')
atomic_write_json(family_path, result)
print('completed pairs:', len(completed), '| binding:', binding)
for row in pairs:
    if row['status'] == 'completed':
        hard = row['tiers']['hard']
        print(f"seed {row['seed']}: hard delta {hard['delta']:+.3f}, "
              f"Holm p {row['hard_holm_p']}, category {row['hard_category']}")
    else:
        print(f"seed {row['seed']}: {row['status']}")
print('common reading:', common)
print('family summary:', family_path)


completed pairs: 2 | binding: False
seed 0: hard delta -0.003, Holm p None, category None
seed 1: viability_gate_failure
seed 2: hard delta -0.025, Holm p None, category None
common reading: None
family summary: /content/drive/MyDrive/inm363/results/030_family_summary.json


## 14. Peek at the result


In [16]:
family_path = os.path.join(D, 'results', '030_family_summary.json')
if os.path.exists(family_path):
    result = json.load(open(family_path))
    print('status:', result['status'])
    print('common reading:', result['common_reading'])
    print('descriptive summary:', result['descriptive_summary'])
    for row in result['pairs']:
        print('seed', row['seed'], row['status'], row.get('hard_category'))
else:
    print('family comparison not available yet')


status: descriptive_incomplete
common reading: None
descriptive summary: {'hard_delta_mean': -0.013888888888888888, 'hard_delta_range': [-0.025252525252525252, -0.0025252525252525255], 'overall_delta_mean': -0.007165605095541401, 'overall_delta_range': [-0.012208067940552018, -0.0021231422505307855], 'all_hard_point_estimates_within_2pt': None}
seed 0 completed None
seed 1 viability_gate_failure None
seed 2 completed None


## 15. Close the units record


In [17]:
# record the balance after the session finishes
import json, os, time
cu_after = None      # fill from the resources panel
if cu_after is None:
    print('reminder: fill cu_after for the record')
elif not os.path.isdir('/content/drive/MyDrive'):
    print('drive not mounted, run the mount cell first')
else:
    os.makedirs(OPEN_UNITS, exist_ok=True)
    open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
    if not os.path.exists(open_path):
        print('no open record for this session, it may already be closed')
    else:
        with open(open_path) as f:
            record = json.load(f)
        record['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        record['colab_cu_after'] = cu_after
        record['colab_cu_used'] = round(record['colab_cu_before'] - cu_after, 2)
        record['closed_on_resume'] = False
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(record) + '\n')
        os.remove(open_path)
        print('logged:', record)


reminder: fill cu_after for the record
